## Section 0 - Drive Mount,Code Import, HF Login , OUT/Cache Dir

In [11]:
# Mount google Drive
import os
from google.colab import drive
drive.mount('/content/drive')

# Add Repo, Clone/pull 
REPO_URL = 'https://github.com/rahulkolayikkath/synthetic-data-pipeline.git'  
%cd /content
if not os.path.exists('/content/synthetic-data-pipeline'):
    !git clone $REPO_URL
%cd /content/synthetic-data-pipeline
!git pull --ff-only

# Code import to working dir
!pip install -q -e .

# OUT_Dir --> out_quick 
os.makedirs('/content/drive/MyDrive/indic_synth/out_quick', exist_ok=True)
OUT = '/content/drive/MyDrive/indic_synth/out_quick'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/synthetic-data-pipeline
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 12 (delta 8), reused 12 (delta 8), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.05 KiB | 161.00 KiB/s, done.
From https://github.com/rahulkolayikkath/synthetic-data-pipeline
   8e40280..5e5b5d6  main       -> origin/main
Updating 8e40280..5e5b5d6
Fast-forward
 config.colab.yaml                          |   7 +-
 config.quick.yaml                          |   5 +-
 config.yaml                                |   5 +-
 notebooks/colab_quick_runbook.ipynb        | 105 ++++++++++++-----------------
 src/indic_synth/tts_generation/__init__.py |   2 +-
 src/indic_synth/tts_generation/config.py   |   6 +-
 6 files changed, 56 insertions(+), 74 deletions(-)
  Installing 

In [12]:
# HF token (kept out of git). Needs: accepted Kathbath terms + accepted Gemma-3 license.
from getpass import getpass
from huggingface_hub import login
os.environ['HF_TOKEN'] = getpass('HF token:')
login(os.environ['HF_TOKEN'])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [13]:
# Add Cache Dir for Gemma (24gb) - only Run cell if you have enough G-drive storage
import os
#os.environ["HF_HOME"] = "/content/drive/MyDrive/indic_synth/hf_cache"
os.environ["HF_HOME"] = "/content/hf_cache"

## Section 1 - 3 (till TTS Generation)

In [14]:
# Install for all stages till tts generation
!pip install -r requirements.txt

In [10]:
!python scripts/run.py --config config.quick.yaml --stages data_acquisition

05:54:42 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
05:54:42 INFO    run | =========== stage: data_acquisition ===========
05:54:42 INFO    run | Acquisition config: {'source': 'hf', 'repo_id': 'ai4bharat/Kathbath', 'local_dir': 'fake_kathbath', 'languages': ['hindi', 'malayalam', 'tamil'], 'split': 'valid', 'speakers_per_language': 2, 'clips_per_speaker': 2, 'min_total_speakers': 4, 'gender_balance': True, 'ref_min_dur': 3.0, 'ref_max_dur': 15.0, 'seed': 1234, 'out_dir': '/content/drive/MyDrive/indic_synth/out_quick', 'hf_token': True, 'max_retries': 4, 'retry_backoff': 2.0, 'force_catalog': False}
05:54:42 INFO    run | Loading cached catalog: /content/drive/MyDrive/indic_synth/out_quick/catalog.parquet
05:54:42 INFO    run | Selected 12 clips across 6 speakers; saved /content/drive/MyDrive/indic_synth/out_quick/selection_manifest.jsonl
05:54:43 INFO    run | 12 selected, 0 already done, 12 to pull
05:54:50 INFO    run | Pull complet

In [4]:
!python scripts/run.py --config config.quick.yaml --stages audio_engineering

07:05:20 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
07:05:20 INFO    run | =========== stage: audio_engineering ===========
07:05:20 INFO    run | 12 downloaded clips, 0 already prepared, 12 to process
07:05:31 INFO    run | Prepare complete: {"stage": "audio_engineering", "elapsed_sec": 10.71, "target_sr": 24000, "norm": "peak", "trim": false, "prepared": 12, "failed": 0, "sr_in": {"16000": 12}, "flags": {"resampled_up": 12}, "backends": {"soundfile": 12}, "prepared_manifest": "/content/drive/MyDrive/indic_synth/out_quick/prepared_manifest.jsonl"}
07:05:32 INFO    run | Requested stages in pipeline done in 11.3s.


In [15]:
!python scripts/run.py --config config.quick.yaml --stages sentence_generation

16:20:31 INFO    run | Pipeline start | out_dir=/content/drive/MyDrive/indic_synth/out_quick seed=1234
16:20:31 INFO    run | =========== stage: sentence_generation ===========
Fetching 5 files: 100% 5/5 [00:00<00:00, 10072.78it/s]
Download complete: : 0.00B [00:00, ?B/s]              
Loading weights:   0% 1/1065 [00:08<2:29:34,  8.43s/it]/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100% 1065/1065 [01:40<00:00, 10.62it/s] 
16:22:32 INFO    run | Model loaded: google/gemma-3-12b-it
Loading weights: 100% 199/199 [00:00<00:00, 761.53it/s]
16:22:50 INFO    run | Resumed 155 sentences from checkpoint.
16:22:50 INFO    run | Grid: 12 cells x quota 10 (~120 target sentences); already have 155.
16:22:50 INFO    run | [1/12] hi|Daily Commute|declarative -> 13/10 valid (0 at

### Section 4 - TTS Generation

In [ ]:
!pip install -q git+https://github.com/ai4bharat/IndicF5.git
!pip install -q "transformers==4.49.0" torch torchaudio soundfile speechbrain jiwer pyyaml

In [ ]:
# Restart The session and Run Section 0 

In [ ]:
# Check transformer Version  # -> 4.49.0
import transformers
print(transformers.__version__)

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages tts_generation

## Section - 5 QC 

In [ ]:
# Update the Versions back
!pip install -r requirements.txt

In [ ]:
# Restart and Run Section zero

In [ ]:
!python scripts/run.py --config config.quick.yaml --stages quality_control

In [ ]:
print(open(f'{OUT}/qc_summary.json').read())

### Sanity Check - Rejected by QC Review

In [ ]:
import json
import os
from typing import Dict, Iterable, Iterator, List
def read_jsonl(path: str) -> List[Dict]:
    """Read a JSONL file into a list of dicts. Missing file -> empty list."""
    if not os.path.exists(path):
        return []
    rows: List[Dict] = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

In [ ]:
# sanity-check thresholds: listen to a couple of QC failures
from IPython.display import Audio, display
rows = read_jsonl(f'{OUT}/dataset_manifest.jsonl')
fails = [r for r in rows if not r['qc_passed']][:] # Edit show much you want to Verify 
print('pass:', sum(r['qc_passed'] for r in rows), '/', len(rows))
for r in fails:
    print(r['utt_id'], 'CER=', r.get('cer'), 'spk=', r.get('speaker_sim'), r['qc_reasons'])
    display(Audio(f"{OUT}/{r['audio_filepath']}"))